In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from torchsummary import summary
import numpy as np
import pandas as pd
import statistics
from openpyxl import Workbook, load_workbook
import os
import random
import matplotlib

In [2]:
from data.audio_preprocessor import AudioPreprocessor
from data.speech_commands_ds import SpeechCommandsDataset
from nets.vgg import VGG1D
from model.training import train_model, plot_training_history, plot_combined_training_history
from model.testing import test_model

In [3]:
# experiments
# 	1d_vgg
# 		summary.xlsx
# 		lr_001
# 			rs_42
# 				analysis.txt
# 				trainig_history.pdf
# 				confusion_matrix.pdf
# 				confusion_matrix_norm.pdf
# 			rs_24
# 			training_history.pdf
# 		lr_01
# 		epochs20

In [4]:
# Base configuration with default values
base_config = {
    'data_dir': 'C:/Users/weron/Pulpit/sem1/dl/proj1/Deep-Learning/transformers/dataset/train',
    'model_type': '1d',
    'model_name': 'vgg',
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'batch_size': 32,          
    'num_epochs': 20,         
    'learning_rate': 0.001,    
    'weight_decay': 0.0001,    
    'title': 'base'
}

experiment_configs = []
experiment_configs.append(base_config)

# Varying batch sizes
batch_sizes = [16, 64, 128]
for bs in batch_sizes:
    config = base_config.copy()
    config.update({
        'batch_size': bs,
        'title': f'batch_{bs}'
    })
    experiment_configs.append(config)

# Varying learning rates
learning_rates = [0.1, 0.01, 0.0001]
for lr in learning_rates:
    config = base_config.copy()
    config.update({
        'learning_rate': lr,
        'title': f'lr_{lr}'
    })
    experiment_configs.append(config)

# Varying weight decay
weight_decays = [0.1, 0.01, 0.001]
for wd in weight_decays:
    config = base_config.copy()
    config.update({
        'weight_decay': wd,
        'title': f'wd_{wd}'
    })
    experiment_configs.append(config)

# Varying epoch counts
epoch_counts = [10, 50, 100]
for epochs in epoch_counts:
    config = base_config.copy()
    config.update({
        'num_epochs': epochs,
        'title': f'epochs_{epochs}'
    })
    experiment_configs.append(config)

In [ ]:
for config in experiment_configs:

    root_dir = os.path.join('experiments', f'{config['model_type']}_{config['model_name']}')
    os.makedirs(root_dir, exist_ok=True)
    
    all_train_acc_hist = []
    all_val_acc_hist = []
    all_train_loss_hist = []
    all_val_loss_hist = []
    run_accuracies = []

    for i in [42, 24, 62]:

        # Set random seed
        random_seed = i
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        torch.cuda.manual_seed(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        np.random.seed(random_seed)

        # Load data
        preprocessor = AudioPreprocessor()

        train_dataset = SpeechCommandsDataset(
            root_dir=config['data_dir'],
            file_list_path='train_list.txt',
            mode=config['model_type'],
            preprocessor=preprocessor,
            augment=False
        )

        val_dataset = SpeechCommandsDataset(
            root_dir=config['data_dir'],
            file_list_path='validation_list.txt',
            mode=config['model_type'],
            preprocessor=preprocessor,
            augment=False
        )

        test_dataset = SpeechCommandsDataset(
            root_dir=config['data_dir'],
            file_list_path='testing_list.txt',
            mode=config['model_type'],
            preprocessor=preprocessor,
            augment=False
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=4
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=4
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=4
        )

        dataloaders = {
            'train': train_loader,
            'valid': val_loader,
            'test': test_loader
        }

        # Create model TODO others
        if config['model_type'] == '1d':
            if config['model_name'] == 'vgg':
                model = VGG1D()
            elif config['model_name'] == 'resnet':
                model = resnet34_1d(num_classes=len(class_to_idx))
        else:  # '2d'
            model = get_2d_cnn(config['model_name'], num_classes=len(class_to_idx))

        model = model.to(config['device'])

        # Train model
        criterion = nn.CrossEntropyLoss()

        optimizer = optim.Adam(model.parameters(), 
                            lr=config['learning_rate'], 
                            weight_decay=config['weight_decay'])
        scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

        trained_model, train_loss_hist, train_acc_hist, val_loss_hist, val_acc_hist = train_model(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            dataloaders=dataloaders,
            device=config['device'],
            num_epochs=config['num_epochs']
        )
        
        # Save best model and model details
        folder = os.path.join(os.path.join(root_dir, config['title'].replace('.', '_')), f'rs_{i}')
        os.makedirs(folder, exist_ok=True)

        torch.save(model.state_dict(), os.path.join(folder, 'best_model.pt'))
        
        # Save training history
        plot_training_history(train_loss_hist, train_acc_hist, val_loss_hist, val_acc_hist, config['title'], folder)
        
        # Test model
        test_acc = test_model(model, test_loader, config['device'], folder, 
            ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'silence', 'stop', 'up', 'yes', 'unknown'])
        
        run_accuracies.append(test_acc)
        all_train_acc_hist.append([acc.cpu().numpy() for acc in train_acc_hist])
        all_val_acc_hist.append([acc.cpu().numpy() for acc in val_acc_hist])
        all_train_loss_hist.append([loss for loss in train_loss_hist])
        all_val_loss_hist.append([loss for loss in val_loss_hist])
        
    # Save three runs summary
    
    # Calculate mean and std
    mean_acc = statistics.mean(run_accuracies)
    std_acc = statistics.stdev(run_accuracies)

    summary_file = os.path.join(root_dir, 'summary.xlsx')
    headers = ['Title', 'Batch Size', 'Learning Rate', 'Weight Decay', 'Epochs',
            'Run 1 Accuracy', 'Run 2 Accuracy', 'Run 3 Accuracy', 'Mean Accuracy', 'Std Accuracy']

    row = [
        config['title'],
        config['batch_size'],
        config['learning_rate'],
        config['weight_decay'],
        config['num_epochs'],
        *run_accuracies,
        mean_acc,
        std_acc
    ]

    # Write row to summary.xlsx
    if not os.path.exists(summary_file):
        wb = Workbook()
        ws = wb.active
        ws.append(headers)
    else:
        wb = load_workbook(summary_file)
        ws = wb.active

    ws.append(row)
    wb.save(summary_file)
    
    plot_combined_training_history(all_train_acc_hist, all_val_acc_hist, all_train_loss_hist, all_val_loss_hist,
                                   config['title'], os.path.join(root_dir, config['title'].replace('.', '_')))
    
    matplotlib.pyplot.close()



Epoch 0/19
----------
train Loss: 1.5387 Acc: 0.6254
valid Loss: 1.4404 Acc: 0.6167

Epoch 1/19
----------
train Loss: 1.0032 Acc: 0.7046
valid Loss: 1.1053 Acc: 0.6457

Epoch 2/19
----------
train Loss: 0.7041 Acc: 0.7975
valid Loss: 2.1024 Acc: 0.6186

Epoch 3/19
----------
train Loss: 0.5104 Acc: 0.8458
valid Loss: 0.7708 Acc: 0.7976

Epoch 4/19
----------
train Loss: 0.3913 Acc: 0.8829
valid Loss: 0.3561 Acc: 0.8856

Epoch 5/19
----------
train Loss: 0.3308 Acc: 0.9016
valid Loss: 0.4755 Acc: 0.8520

Epoch 6/19
----------
train Loss: 0.2936 Acc: 0.9147
valid Loss: 0.3122 Acc: 0.9061

Epoch 7/19
----------
train Loss: 0.1728 Acc: 0.9496
valid Loss: 0.1860 Acc: 0.9418

Epoch 8/19
----------
train Loss: 0.1532 Acc: 0.9551
valid Loss: 0.1825 Acc: 0.9456

Epoch 9/19
----------
train Loss: 0.1402 Acc: 0.9583
valid Loss: 0.1883 Acc: 0.9440

Epoch 10/19
----------
train Loss: 0.1322 Acc: 0.9609
valid Loss: 0.1733 Acc: 0.9472

Epoch 11/19
----------
train Loss: 0.1269 Acc: 0.9624
valid Loss

C:\Users\weron\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_plot\confusion_matrix.py:140: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots()


Epoch 0/19
----------
train Loss: 1.5972 Acc: 0.6228
valid Loss: 1.1660 Acc: 0.6383

Epoch 1/19
----------
train Loss: 0.9460 Acc: 0.7161
valid Loss: 0.6723 Acc: 0.7761

Epoch 2/19
----------
train Loss: 0.5512 Acc: 0.8316
valid Loss: 0.6370 Acc: 0.7888

Epoch 3/19
----------
train Loss: 0.4188 Acc: 0.8775
valid Loss: 0.3759 Acc: 0.8807

Epoch 4/19
----------
train Loss: 0.3976 Acc: 0.8882
valid Loss: 2.6059 Acc: 0.6449

Epoch 5/19
----------
train Loss: 0.3107 Acc: 0.9111
valid Loss: 0.2812 Acc: 0.9118

Epoch 6/19
----------
train Loss: 0.2904 Acc: 0.9173
valid Loss: 0.4276 Acc: 0.8599

Epoch 7/19
----------
train Loss: 0.1690 Acc: 0.9531
valid Loss: 0.1831 Acc: 0.9424

Epoch 8/19
----------
train Loss: 0.1444 Acc: 0.9575
valid Loss: 0.1828 Acc: 0.9456

Epoch 9/19
----------
train Loss: 0.1332 Acc: 0.9612
valid Loss: 0.1733 Acc: 0.9476

Epoch 10/19
----------
train Loss: 0.1272 Acc: 0.9630
valid Loss: 0.1712 Acc: 0.9493

Epoch 11/19
----------
train Loss: 0.1197 Acc: 0.9647
valid Loss

In [ ]:
# from torchsummary import summary

# summary(model, (1, 16384))

# model = model.to(config['device'])
# sample_input = torch.randn(1, 1, 16384).to(config['device'])
# model.eval()
# with torch.no_grad():
#     output = model(sample_input)
# print(output.shape)
